# 04 — Evaluation (Model evaluation)

Evaluates the deep-learning and transformer models, then compares everything:

- **CNN** — 10-fold CV, averaged learning curves
- **LSTM** — bidirectional, 10-fold CV
- **BERTweet** — pre-trained `NLP-LTU/bertweet-large-sexism-detector` on the held-out test set
- **Naive Bayes + SMOTE** — recall-focused variant with threshold adjustment
- **SHAP** — token-level explainability for the LSTM

Run `03_modeling.ipynb` first (or its cells) — this notebook assumes the same variables in scope.

## Imports & setup

In [ ]:
!pip install nltk
!pip show nltk
!pip install tldextract
!pip install datasets
!pip install shap

In [ ]:
import os
import requests
import json
import time
import zipfile

import nltk
import kagglehub
import re

import matplotlib as mpl
import matplotlib.pyplot as plt

import seaborn as sns
import pandas as pd
import numpy as np

from collections import Counter

import tldextract

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer


nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.sparse import hstack
from sklearn.svm import SVC

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback
from tensorflow.keras.mixed_precision import set_global_policy
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, LSTM, Dense, SpatialDropout1D, Dropout, BatchNormalization, Bidirectional, Attention
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from keras.metrics import AUC, Precision, Recall

import torch
from torch.utils.data import DataLoader
from torch.nn import CrossEntropyLoss
from torch.optim import AdamW
from tensorflow.keras.optimizers import Adam
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

from imblearn.over_sampling import SMOTE
import shap

In [ ]:
import sys, os

PROJECT_PATH = "/content/Sexism-Classification" if os.path.exists("/content/Sexism-Classification") else os.getcwd()
sys.path.append(PROJECT_PATH)

In [ ]:
# Download latest version
path = kagglehub.dataset_download("aadyasingh55/sexism-detection-in-english-texts")


print("Path to dataset files:", path)

dev_df = pd.read_csv(f"{path}/dev.csv")
test_df = pd.read_csv(f"{path}/test (1).csv")
train_df = pd.read_csv(f"{path}/train (2).csv")


In [ ]:
# Initialise NLP tools
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Function to clean text and extract domains
def clean_text(text, lowercase=True, replace_urls=True, extract_domain=False, remove_stopwords=True, lemmatize=True):
    if lowercase:
        text = text.lower()

    # Extract domain names from URLs
    url_pattern = r'https?://\S+|www\.\S+'
    domains = []  # Store extracted domains
    matches = re.findall(url_pattern, text)

    for match in matches:
        extracted = tldextract.extract(match)
        domain = f"{extracted.domain}.{extracted.suffix}"  # e.g., "cnn.com"
        domains.append(domain)  # Save domain for analysis
        text = text.replace(match, "")  # Remove the URL from text

    # Remove special characters, punctuation, and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords
    if remove_stopwords:
        tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    if lemmatize:
        tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens), domains  # Convert tokens back to string


In [ ]:
# Apply function to dataset
train_dev_data[["text", "domains"]] = pd.DataFrame(train_dev_data["text"].apply(clean_text).tolist(), index=train_dev_data.index)

# Process the separate test set as well
test_df[["text", "domains"]] = pd.DataFrame(test_df["text"].apply(clean_text).tolist(), index=test_df.index)

print(train_dev_data.head())


# Ensures mixed precision is set globally (allows for 16bit and 32bit float types (runs faster uses less memory))
set_global_policy('mixed_float16')

In [ ]:
# Initialize MultiLabelBinarizer for domains (fit on combined train_dev_data)
mlb = MultiLabelBinarizer()
domain_features_train = mlb.fit_transform(train_dev_data["domains"])

# Vectorize the cleaned text using TF-IDF (fit on combined train_dev_data)
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
X_text_train = vectorizer.fit_transform(train_dev_data["text"])
X_combined_train = hstack([X_text_train, domain_features_train])

# Encode the target variable for deep learning models
label_encoder = LabelEncoder()
y_encoded_train = label_encoder.fit_transform(train_dev_data['label_sexist'])


# --- Prepare the separate TEST set features ---
domain_features_test = mlb.transform(test_df["domains"]) # Use fitted MLB
X_text_test = vectorizer.transform(test_df["text"]) # Use fitted TF-IDF Vectorizer
X_combined_test = hstack([X_text_test, domain_features_test])
y_encoded_test = label_encoder.transform(test_df['label_sexist'])


# Tokenize for deep learning models (fit on combined train_dev_data)
tokenizer = Tokenizer(num_words=5000, oov_token='<OOV>')
tokenizer.fit_on_texts(train_dev_data['text'])
X_seq_train = tokenizer.texts_to_sequences(train_dev_data['text'])

VOCAB_SIZE_DEEP_LEARNING = len(tokenizer.word_index) + 1
if tokenizer.num_words is not None:
    VOCAB_SIZE_DEEP_LEARNING = min(VOCAB_SIZE_DEEP_LEARNING, tokenizer.num_words + 1)

# Determine maxlen for padding based on train_dev_data
percentile_95 = int(np.percentile([len(seq) for seq in X_seq_train], 95))
MAX_SEQUENCE_LENGTH = percentile_95
X_pad_train = pad_sequences(X_seq_train, maxlen=MAX_SEQUENCE_LENGTH, padding='post')

# Tokenize and pad the separate TEST set for deep learning models
X_seq_test = tokenizer.texts_to_sequences(test_df['text'])
X_pad_test = pad_sequences(X_seq_test, maxlen=MAX_SEQUENCE_LENGTH, padding='post')




In [ ]:
# Define common variables for cross-validation
n_splits = 10 # Number of folds for StratifiedKFold
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42) # Set random_state for reproducibility of splits

# Initialise results
results = {
  "LR_Text": [],
  "LR_Domains": [],
  "NB": [],
  "RF": [],
  "SVM": [],
  "LSTM": [],
  "CNN": [],
  "BERT": [],
  "NBft": [],
}

# Initialize history storage for deep learning models (for plotting average curves)
avg_histories = {
    "CNN": {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': [],
        'precision': [], 'recall': [], 'val_precision': [], 'val_recall': [],
        'learning_rate': []},
    "LSTM": {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': [],
        'precision': [], 'recall': [], 'val_precision': [], 'val_recall': [],
        'learning_rate': []},
    "BERT": {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": [], "learning_rate": []},
}

# Initialize lists for additional metrics for each model
precision_list_lr_text, recall_list_lr_text, f1_list_lr_text = [], [], []
precision_list_lr_combined, recall_list_lr_combined, f1_list_lr_combined = [], [], []
precision_list_nb, recall_list_nb, f1_list_nb = [], [], []
precision_list_rf, recall_list_rf, f1_list_rf = [], [], []
precision_list_svm, recall_list_svm, f1_list_svm = [], [], []
precision_list_cnn, recall_list_cnn, f1_list_cnn = [], [], []
precision_list_lstm, recall_list_lstm, f1_list_lstm = [], [], []
precision_list_bert, recall_list_bert, f1_list_bert = [], [], []
precision_list_nbft, recall_list_nbft, f1_list_nbft = [], [], []

# Initialize confusion matrix sums (ensure they are reset or properly scoped for each model)
conf_matrix_sum_lr_text = np.zeros((2, 2))
conf_matrix_sum_lr_combined = np.zeros((2, 2))
conf_matrix_sum_nb = np.zeros((2, 2))
conf_matrix_sum_rf = np.zeros((2, 2))
conf_matrix_sum_svm = np.zeros((2, 2))
conf_matrix_sum_cnn = np.zeros((2, 2))
conf_matrix_sum_lstm = np.zeros((2, 2))
conf_matrix_sum_bert = np.zeros((2, 2))
conf_matrix_sum_nbft = np.zeros((2, 2))


## Deep Learning models

### CNN

In [ ]:
# Custom Learning Rate Logger (Optional but useful)
class LearningRateLogger(Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        lr = 0.0 # Default value

        # Trying to get the learning rate from the inner_optimizer
        if hasattr(self.model.optimizer, 'inner_optimizer'):
            # Access learning rate from the inner optimizer's 'learning_rate' attribute
            if hasattr(self.model.optimizer.inner_optimizer, 'learning_rate'):
                lr = tf.keras.backend.get_value(self.model.optimizer.inner_optimizer.learning_rate)
            elif hasattr(self.model.optimizer.inner_optimizer, 'lr'): # Fallback for older .lr name
                lr = tf.keras.backend.get_value(self.model.optimizer.inner_optimizer.lr)
        # If no inner_optimizer, try to get it directly from the optimizer
        elif hasattr(self.model.optimizer, 'learning_rate'):
            lr = tf.keras.backend.get_value(self.model.optimizer.learning_rate)
        elif hasattr(self.model.optimizer, 'lr'):
            lr = tf.keras.backend.get_value(self.model.optimizer.lr)
        else:
            # Fallback if no known attribute is found, though this should be rare for standard optimizers
            print(f"Warning: Could not determine learning rate for optimizer type {type(self.model.optimizer).__name__}")

        logs['learning_rate'] = lr


# Function to Define CNN Model (for reusability)
def build_cnn_model(vocab_size, max_sequence_length, learning_rate=0.00008):
    model = Sequential()
    model.add(Embedding(input_dim=vocab_size, output_dim=128, input_length=max_sequence_length))
    model.add(Conv1D(filters=32, kernel_size=5, activation='relu', kernel_regularizer=l2(0.001)))
    model.add(BatchNormalization())
    model.add(GlobalMaxPooling1D())
    model.add(Dropout(0.4))
    model.add(Dense(32, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(16, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='sigmoid', dtype='float32')) # Ensure float32 for output

    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy',
                  metrics=['accuracy', Precision(name='precision'), Recall(name='recall')])
    return model

In [ ]:
print("\n--- Training CNN Model ---")

# K-Fold Loop for CNN
for fold, (train_index, test_index) in enumerate(skf.split(X_pad_train, y_encoded_train)):
    print(f"\n CNN - Fold {fold + 1}/{n_splits}...")

    X_train_cnn, X_test_cnn = X_pad_train[train_index], X_pad_train[test_index]
    y_train_cnn, y_test_cnn = y_encoded_train[train_index], y_encoded_train[test_index]

    # Debugging prints
    # print(f"  X_train_cnn shape: {X_train_cnn.shape}, dtype: {X_train_cnn.dtype}")
    # print(f"  y_train_cnn shape: {y_train_cnn.shape}, dtype: {y_train_cnn.dtype}")
    # print(f"  Max value in X_train_cnn: {np.max(X_train_cnn)}") # Should be less than input_dim
    # print(f"  Min value in X_train_cnn: {np.min(X_train_cnn)}") # Should be 0 or positive
    # print(f"  Expected Embedding input_dim: {VOCAB_SIZE_DEEP_LEARNING}")
    # print(f"  Expected Embedding input_length: {MAX_SEQUENCE_LENGTH}")


    # Define CNN model (re-initialize for each fold to ensure fresh weights)
    cnn_model = build_cnn_model(VOCAB_SIZE_DEEP_LEARNING, MAX_SEQUENCE_LENGTH)

    # Print Model Summary (only once, for the first fold)
    if fold == 0:
        print("\n--- CNN Model Summary ---")
        cnn_model.summary()
        print("-" * 30)

    # Callbacks
    early_stopping = EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
    lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.7, patience=3, verbose=0, min_lr=1e-5)
    lr_logger_callback = LearningRateLogger() # To get the learning rate at each fold

    # Train the model (Epochs for training over each fold)
    cnn_history = cnn_model.fit(X_train_cnn, y_train_cnn,
                                 epochs=10,
                                 batch_size=128,
                                 callbacks=[early_stopping, lr_scheduler, lr_logger_callback],
                                 validation_data=(X_test_cnn, y_test_cnn),
                                 verbose=1)

    # Model Performance (Accuracy, Precision, Recall, F1) per fold
    # Returns: loss, accuracy, precision and recall
    cnn_loss, cnn_acc, cnn_precision, cnn_recall = cnn_model.evaluate(X_test_cnn, y_test_cnn, verbose=0)

    # Calculate F1-score
    cnn_f1 = 2 * (cnn_precision * cnn_recall) / (cnn_precision + cnn_recall + 1e-7) # Add epsilon to avoid div by zero

    results["CNN"].append(cnn_acc)

    print(f"    CNN Fold {fold + 1} Metrics - Accuracy: {cnn_acc:.4f}, Precision: {cnn_precision:.4f}, Recall: {cnn_recall:.4f}, F1: {cnn_f1:.4f}")

    # Confusion Matrix per fold
    y_pred_probs_cnn = cnn_model.predict(X_test_cnn, verbose=0)
    y_pred_cnn = (y_pred_probs_cnn > 0.8).astype(int)
    current_cm = confusion_matrix(y_test_cnn, y_pred_cnn)
    conf_matrix_sum_cnn += current_cm # Accumulate for averaging


    # Accumulate history for averaging later
    # Ensure all metrics from cnn_history.history are captured
    for metric_name, values in cnn_history.history.items():
        if metric_name in avg_histories["CNN"]:
            avg_histories["CNN"][metric_name].append(values)
        else:
            print(f"Warning: Metric '{metric_name}' not pre-initialized for CNN history.")

# Post-Loop Processing for Averaged Results and Plots

# Average CNN histories
print("\n--- Averaging CNN Histories ---")
for metric, all_runs_values in avg_histories["CNN"].items():

    # Pad shorter histories to the minimum length (due to EarlyStopping)
    min_len = min(len(run_values) for run_values in all_runs_values)
    trimmed_values = [run_values[:min_len] for run_values in all_runs_values]
    avg_histories["CNN"][metric] = np.mean(trimmed_values, axis=0)

# Calculate and print average results for CNN
avg_accuracy_cnn = np.mean(results["CNN"])
std_accuracy_cnn = np.std(results["CNN"])
print(f"\nAverage Accuracy for CNN over {n_splits} folds: {avg_accuracy_cnn:.4f} ± {std_accuracy_cnn:.4f}")

# Averaged Confusion Matrix Plot
print("\n--- Averaged CNN Confusion Matrix ---")
avg_cm = conf_matrix_sum_cnn / n_splits # Average the accumulated confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(avg_cm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title(f'Averaged CNN Confusion Matrix over {n_splits} Folds')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()

# Averaged Plots for Model Accuracy and Model Loss over All Epochs
print("\n--- Averaged CNN Learning Curves ---")
plt.figure(figsize=(12, 5))

# Plot Average Accuracy
plt.subplot(1, 2, 1)
plt.plot(avg_histories["CNN"]['accuracy'], label='CNN Train Accuracy')
plt.plot(avg_histories["CNN"]['val_accuracy'], label='CNN Val Accuracy')
plt.title(f'CNN Average Accuracy over {n_splits} Folds')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot Average Loss
plt.subplot(1, 2, 2)
plt.plot(avg_histories["CNN"]['loss'], label='CNN Train Loss')
plt.plot(avg_histories["CNN"]['val_loss'], label='CNN Val Loss')
plt.title(f'CNN Average Loss over {n_splits} Folds')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Optional: Plot Average Precision/Recall
# plt.figure(figsize=(12, 5))
# plt.subplot(1, 2, 1)
# plt.plot(avg_histories["CNN"]['precision'], label='CNN Train Precision')
# plt.plot(avg_histories["CNN"]['val_precision'], label='CNN Val Precision')
# plt.title(f'CNN Average Precision over {n_splits} Folds')
# plt.xlabel('Epochs')
# plt.ylabel('Precision')
# plt.legend()
# plt.grid(True)
# plt.subplot(1, 2, 2)
# plt.plot(avg_histories["CNN"]['recall'], label='CNN Train Recall')
# plt.plot(avg_histories["CNN"]['val_recall'], label='CNN Val Recall')
# plt.title(f'CNN Average Recall over {n_splits} Folds')
# plt.xlabel('Epochs')
# plt.ylabel('Recall')
# plt.legend()
# plt.grid(True)
# plt.tight_layout()
# plt.show()

### LSTM

In [ ]:
shap_values_list = []
test_data_list = []

In [ ]:
conf_matrix_sum_lstm = np.zeros((2, 2))

for fold, (train_index, test_index) in enumerate(skf.split(X_pad_train, y_encoded_train)):
    print(f"LSTM - Fold {fold + 1}/{n_splits}...")

    X_train_lstm, X_test_lstm = X_pad_train[train_index], X_pad_train[test_index]
    y_train_lstm, y_test_lstm = y_encoded_train[train_index], y_encoded_train[test_index]

    # Define LSTM model (re-initialize for each fold)
    lstm_model = Sequential([
        Embedding(input_dim=VOCAB_SIZE_DEEP_LEARNING, output_dim=128),
        SpatialDropout1D(0.2),
        Bidirectional(LSTM(64, return_sequences=True)),
        LSTM(64),
        Dense(1, activation='sigmoid', dtype='float32')
    ])

    if fold == 0:
        print("\n--- LSTM Model Summary ---")
        lstm_model.summary()
        print("-" * 30)

    # Compile LSTM model with Precision and Recall metrics
    lstm_model.compile(optimizer=Adam(learning_rate=0.0009),
                      loss='binary_crossentropy',
                      metrics=['accuracy', Precision(name='precision'), Recall(name='recall')])

    early_stopping = EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
    lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.7, patience=3, verbose=0, min_lr=1e-5)
    lr_logger_callback = LearningRateLogger()

    lstm_history = lstm_model.fit(X_train_lstm, y_train_lstm, epochs=5, batch_size=128,
                                  callbacks=[early_stopping, lr_scheduler, lr_logger_callback],
                                  validation_data=(X_test_lstm, y_test_lstm), verbose=1)


    #Required for SHAP analysis
    # Initialize the DeepExplainer with the trained model and its corresponding background data
    explainer = shap.KernelExplainer(lstm_model, X_train_lstm)

    # Select some test data from this fold to explain
    X_test_to_explain = X_test_lstm[:10]

    # Calculate SHAP values for the selected examples from this fold
    shap_values = explainer.shap_values(X_test_to_explain)

    # You can now process or store the shap_values for this fold
    shap_values_list.append(shap_values)
    test_data_list.append(X_test_to_explain)

    lstm_loss, lstm_acc, lstm_precision, lstm_recall = lstm_model.evaluate(X_test_lstm, y_test_lstm, verbose=0)

    lstm_f1 = 2 * (lstm_precision * lstm_recall) / (lstm_precision + lstm_recall + 1e-7)

    results["LSTM"].append(lstm_acc)

    print(f"    LSTM Fold {fold + 1} Metrics - Accuracy: {lstm_acc:.4f}, Precision: {lstm_precision:.4f}, Recall: {lstm_recall:.4f}, F1: {lstm_f1:.4f}")

    y_pred_probs_lstm = lstm_model.predict(X_test_lstm, verbose=0)
    y_pred_lstm = (y_pred_probs_lstm > 0.8).astype(int)
    current_cm = confusion_matrix(y_test_lstm, y_pred_lstm)
    # Renamed from cm_sums to conf_matrix_sum_lstm
    conf_matrix_sum_lstm += current_cm

    for metric_name, values in lstm_history.history.items():
        if metric_name in avg_histories["LSTM"]:
            avg_histories["LSTM"][metric_name].append(values)
        else:
           print(f"Warning: Metric '{metric_name}' not pre-initialized for LSTM history.")

# The rest of the code for averaging and plotting remains correct
print("\n--- Averaging LSTM Histories ---")
for metric, all_runs_values in avg_histories["LSTM"].items():
   min_len = min(len(run_values) for run_values in all_runs_values)
   trimmed_values = [run_values[:min_len] for run_values in all_runs_values]
   avg_histories["LSTM"][metric] = np.mean(trimmed_values, axis=0)

avg_accuracy_lstm = np.mean(results["LSTM"])
std_accuracy_lstm = np.std(results["LSTM"])
print(f"\nAverage Accuracy for LSTM over {n_splits} folds: {avg_accuracy_lstm:.4f} ± {std_accuracy_lstm:.4f}")

print("\n--- Averaged LSTM Confusion Matrix ---")
# Renamed from cm_sums to conf_matrix_sum_lstm
avg_cm_lstm = conf_matrix_sum_lstm / n_splits
plt.figure(figsize=(6, 5))
sns.heatmap(avg_cm_lstm, annot=True, fmt=".2f", cmap="Blues",
           xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title(f'Averaged LSTM Confusion Matrix over {n_splits} Folds')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()

print("\n--- Averaged LSTM Learning Curves ---")
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(avg_histories["LSTM"]['accuracy'], label='LSTM Train Accuracy')
plt.plot(avg_histories["LSTM"]['val_accuracy'], label='LSTM Val Accuracy')
plt.title(f'LSTM Average Accuracy over {n_splits} Folds')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(avg_histories["LSTM"]['loss'], label='LSTM Train Loss')
plt.plot(avg_histories["LSTM"]['val_loss'], label='LSTM Val Loss')
plt.title(f'LSTM Average Loss over {n_splits} Folds')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### Pre-Trained BERTweet

In [ ]:
# Initializing BERTweet Sexism Detector
print("--- Initializing BERTweet Sexism Detector ---")

tokenizer = AutoTokenizer.from_pretrained("NLP-LTU/bertweet-large-sexism-detector")
model = AutoModelForSequenceClassification.from_pretrained("NLP-LTU/bertweet-large-sexism-detector")

# Create a Hugging Face pipeline for text classification.
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

# Encode the true labels from test_df for comparison
y_true_encoded = label_encoder.transform(test_df['label_sexist'])

#  Performing Predictions on the Test Dataset
print("\n--- Performing Predictions on the Test Dataset ---")

# Get predictions for all texts in the test_df
# The pipeline returns a list of dictionaries, so we extract the 'label'
texts_to_predict = test_df['text'].tolist()
print(f"Number of texts to predict: {len(texts_to_predict)}")

predictions_raw = classifier(texts_to_predict)
print(f"Length of predictions_raw (from classifier): {len(predictions_raw)}")
print(f"First 5 raw predictions: {predictions_raw[:5]}") # Debug print

# Extract predicted labels and scores
predicted_labels_raw = [p['label'] for p in predictions_raw]
prediction_scores = [p['score'] for p in predictions_raw]
print(f"Length of predicted_labels_raw (extracted labels): {len(predicted_labels_raw)}")
print(f"First 5 extracted labels: {predicted_labels_raw[:5]}") # Debug print


# Map the generic labels ('LABEL_0', 'LABEL_1') to your human-readable labels ('not sexist', 'sexist')
# Based on the model card for 'NLP-LTU/bertweet-large-sexism-detector',
# it's highly likely the model directly outputs 'not sexist' and 'sexist' strings.
# ive directly used these labels as they match what label_encoder expects.
y_pred_mapped = predicted_labels_raw # Direct assignment, assuming labels are already in desired format

print(f"Length of y_pred_mapped (after mapping): {len(y_pred_mapped)}")
print(f"First 5 mapped labels: {y_pred_mapped[:5]}") # Debug print

y_pred_encoded = label_encoder.transform(y_pred_mapped)
print(f"Length of y_pred_encoded (after label encoding): {len(y_pred_encoded)}")


# Evaluate Model Performance
print("\n--- BERTweet Model Performance on Test Set ---")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_true_encoded, y_pred_encoded, target_names=label_encoder.classes_))

# Confusion Matrix
cm = confusion_matrix(y_true_encoded, y_pred_encoded)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title('BERTweet Sexism Detector Confusion Matrix on Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

print("\nBERTweet Model evaluation on the test dataset complete.")

### Fine-tuning BERTweet (vinai/bertweet-base)

The API's lifespan loads `vinai/bertweet-base` with a **fresh classification head** (randomly initialised — see `src/api/model_manager.py`). This section fine-tunes that exact model on the EDOS training data and saves the checkpoint to `LOCAL_MODEL_DIR` (`outputs/models/bertweet-sexism` by default). **The API picks it up automatically on next restart** — no code changes needed.

Uses the dissertation's hyperparameters from `src/config/settings.py`: `NUM_EPOCHS=3`, `LEARNING_RATE=2e-5`, `BATCH_SIZE=16`, `MAX_LENGTH=128`. GPU strongly recommended (mixed precision is already enabled above).

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from src.config.settings import (
    MODEL_ID, LOCAL_MODEL_DIR, NUM_EPOCHS, LEARNING_RATE, BATCH_SIZE, MAX_LENGTH, RANDOM_SEED,
)
import copy

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training on: {device}")

# Download pre-trained Transformer weights and attach a FRESH classification head
# (same call the API lifespan makes — deterministic pairing with src/api/model_manager.py)
ft_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)
ft_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
ft_model.to(device)
print("Fresh head initialised:", ft_model.classifier)

In [ ]:
from torch.utils.data import Dataset

class SexismDataset(Dataset):
    """Tokenises on the fly so the full corpus never sits in memory."""
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_length = tokenizer, max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, max_length=self.max_length,
            padding='max_length', return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
        }

# train_dev_data / label_encoder come from the pipeline cells above
ft_train = SexismDataset(
    train_dev_data['text'].tolist(),
    label_encoder.transform(train_dev_data['label_sexist']),
    ft_tokenizer,
)
from torch.utils.data import DataLoader
ft_loader = DataLoader(ft_train, batch_size=BATCH_SIZE, shuffle=True)
print(f"Training samples: {len(ft_train)} | batches/epoch: {len(ft_loader)}")

In [ ]:
from torch.nn import CrossEntropyLoss
from torch.optim import AdamW

optimizer = AdamW(ft_model.parameters(), lr=LEARNING_RATE)
loss_fn = CrossEntropyLoss()

ft_model.train()
for epoch in range(NUM_EPOCHS):
    running_loss, correct, seen = 0.0, 0, 0

    for step, batch in enumerate(ft_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        logits = ft_model(input_ids=input_ids, attention_mask=attention_mask).logits
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        correct += (logits.argmax(dim=-1) == labels).sum().item()
        seen += labels.size(0)

        if (step + 1) % 200 == 0:
            print(f"  epoch {epoch+1}/{NUM_EPOCHS} step {step+1}: loss={loss.item():.4f}")

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} — avg loss: {running_loss / (step + 1):.4f}, train acc: {correct / seen:.4f}")

print("Fine-tuning complete.")

In [ ]:
ft_model.eval()
ft_correct, ft_seen = 0, 0
ft_loader_eval = DataLoader(ft_train, batch_size=BATCH_SIZE)  # train-set sanity check

with torch.no_grad():
    for batch in ft_loader_eval:
        logits = ft_model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
        ).logits
        ft_correct += (logits.argmax(dim=-1) == batch['labels']).sum().item()
        ft_seen += batch['labels'].shape[0]

print(f"Training accuracy after fine-tuning: {ft_correct / ft_seen:.4f}")

In [ ]:
# Save the checkpoint exactly where the API lifespan expects it
# (LOCAL_MODEL_DIR — settings default: outputs/models/bertweet-sexism)
import os
os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)

ft_model.save_pretrained(LOCAL_MODEL_DIR)
ft_tokenizer.save_pretrained(LOCAL_MODEL_DIR)
print(f"Checkpoint saved to {LOCAL_MODEL_DIR}")
print(os.listdir(LOCAL_MODEL_DIR))

#### Deploy the fine-tuned model

```bash
# 1. Restart the API — the lifespan handler detects the checkpoint at
#    LOCAL_MODEL_DIR and loads it instead of the fresh-headed base model:
uvicorn src.api.fastapi_main:app --reload

# 2. Confirm /health reports fresh_head: false (fine-tuned checkpoint loaded):
curl http://localhost:8000/health
```

#Bias Fine-tuning

### Bias fine-tuning (SMOTE + recall-focused Naive Bayes)

In [ ]:
def apply_smote(X_train, y_train):
    # Ensure X_train is not empty before applying SMOTE
    if X_train.shape[0] > 0 and len(np.unique(y_train)) > 1:
        smote = SMOTE(random_state=42)
        # Use a try-except block in case SMOTE fails on a particular fold
        try:
            X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
            return X_resampled, y_resampled
        except ValueError as e:
            print(f"Warning: SMOTE could not be applied to this fold. Skipping resampling. Error: {e}")
            return X_train, y_train
    else:
        print("Warning: Training data for this fold is empty or contains only one class. Skipping SMOTE.")
        return X_train, y_train


for fold, (train_index, test_index) in enumerate(skf.split(X_text_train, y_encoded_train)):
    print(f"\nNaive Bayes (Recall-focused) - Fold {fold + 1}/{n_splits}...")

    # Ensure X_text_train is in CSR format
    X_text_train_csr = X_text_train.tocsr()

    X_train_nbft, X_test_nbft = X_text_train_csr[train_index], X_text_train_csr[test_index]
    y_train_nbft, y_test_nbft = y_encoded_train[train_index], y_encoded_train[test_index]

    # --- Apply SMOTE to the training data of the current fold ---
    X_train_resampled, y_train_resampled = apply_smote(X_train_nbft, y_train_nbft)

    # Check shape after resampling
    print(f"  Training data shape before SMOTE: {X_train_nbft.shape}, Minority class count: {np.sum(y_train_nbft == 1)}")
    print(f"  Training data shape after SMOTE: {X_train_resampled.shape}, Minority class count: {np.sum(y_train_resampled == 1)}")

    # --- Use class_prior=[0.5, 0.5] in the model ---
    nbft_model = MultinomialNB(class_prior=[0.5, 0.5])
    nbft_model.fit(X_train_resampled, y_train_resampled)

    y_pred_nbft = nbft_model.predict(X_test_nb)
    acc_nbft = accuracy_score(y_test_nbft, y_pred_nbft)
    results["NBft"].append(acc_nbft)

    precision_nbft_f, recall_nbft_f, f1_nbft_f, _ = precision_recall_fscore_support(y_test_nbft, y_pred_nbft, average='binary', pos_label=1)
    precision_list_nbft.append(precision_nbft_f)
    recall_list_nbft.append(recall_nbft_f)
    f1_list_nbft.append(f1_nbft_f)
    conf_matrix_sum_nbft += confusion_matrix(y_test_nbft, y_pred_nbft)

    print(f"    Recall-focused NBft - Accuracy: {acc_nbft:.4f}, Precision: {precision_nbft_f:.4f}, Recall: {recall_nbft_f:.4f}, F1 Score: {f1_nbft_f:.4f}")

    # --- Threshold Adjustment Demonstration ---
    # This part shows how to manually adjust the threshold to prioritize recall
    y_pred_probs = nbft_model.predict_proba(X_test_nbft)[:, 1]
    threshold = 0.3
    y_pred_nbft_recall_tuned = (y_pred_probs > threshold).astype(int)

    precision_nbft_f_tuned, recall_nbft_f_tuned, f1_nbft_f_tuned, _ = precision_recall_fscore_support(y_test_nbft, y_pred_nbft_recall_tuned, average='binary', pos_label=1)

    print(f"--- Results with Threshold Adjusted to {threshold} ---")
    print(f"Tuned Precision: {precision_nbft_f_tuned:.4f}, Tuned Recall: {recall_nbft_f_tuned:.4f}, Tuned F1 Score: {f1_nbft_f_tuned:.4f}")

# Calculate and print average results for the recall-focused NBft
avg_accuracy_nbft = np.mean(results["NBft"])
std_accuracy_nbft = np.std(results["NBft"])
avg_precision_nbft = np.mean(precision_list_nbft)
avg_recall_nbft = np.mean(recall_list_nbft)
avg_f1_nbft = np.mean(f1_list_nbft)

print(f"\nAverage Accuracy for Recall-focused NBft over {n_splits} folds: {avg_accuracy_nbft:.4f} ± {std_accuracy_nbft:.4f}")
print(f"    Avg Precision: {avg_precision_nbft:.4f}, Avg Recall: {avg_recall_nbft:.4f}, Avg F1 Score: {avg_f1_nbft:.4f}")

# Plotting average confusion matrix for recall-focused NBft
avg_conf_matrix_nbft = conf_matrix_sum_nbft / n_splits
conf_matrix_custom_nbft = np.array([[avg_conf_matrix_nbft[1, 1], avg_conf_matrix_nbft[0, 1]],
                                         [avg_conf_matrix_nb[1, 0], avg_conf_matrix_nbft[0, 0]]], dtype=int)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_custom_nbft, annot=True, fmt="d", cmap="Blues",
            yticklabels=["Actual Sexist", "Actual Not Sexist"],
            xticklabels=["Predicted Sexist", "Predicted Not Sexist"])
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Recall-focused Naive Bayes Average Confusion Matrix")
plt.show()
print("Recall-focused Naive Bayes Average Confusion Matrix:\n", conf_matrix_custom_nbft)

## SHAP Explainability

In [ ]:
# Initialize the DeepExplainer with the model and background data
explainer = shap.DeepExplainer(lstm_model, X_pad_train)

# Select data to explain (the first 10)
X_test_to_explain = X_pad_test[:10]

# Calculate SHAP values for the selected examples
shap_values = explainer.shap_values(X_test_to_explain)

#Define the mapping from token IDs back to words
word_index = tokenizer.word_index
index_to_word = {v: k for k, v in word_index.items()}

#Define the helper function to map SHAP values to words
def get_words_from_ids(ids):
    # This will return a list of words corresponding to the token IDs
    return [index_to_word.get(i, "") for i in ids]

# Select the data point you want to explain
test_instance_index = 0
test_data_to_explain = X_pad_test[test_instance_index:test_instance_index+1]

# 4. Calculate the SHAP values for this specific data point
shap_values = explainer.shap_values(test_data_to_explain)

# 5. Generate and display the waterfall plot
# This plot will show the contribution of each word to the model's output
shap.plots.waterfall(shap.Explanation(
    values=shap_values[0][0], # SHAP values for the first class of the first instance
    base_values=explainer.expected_value[0],
    data=test_data_to_explain[0],
    feature_names=get_words_from_ids(test_data_to_explain[0])
))

## Evaluating the Models

In [ ]:
# Plot CNN
if "CNN" in avg_histories and avg_histories["CNN"]["accuracy"].size > 0: # Check if data exists
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(avg_histories["CNN"]['accuracy'], label='CNN Train Acc')
    plt.plot(avg_histories["CNN"]['val_accuracy'], label='CNN Val Acc')
    plt.title('CNN Average Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(avg_histories["CNN"]['loss'], label='CNN Train Loss')
    plt.plot(avg_histories["CNN"]['val_loss'], label='CNN Val Loss')
    plt.title('CNN Average Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.suptitle("CNN Averaged Learning Curves Across Folds", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
    plt.show()

# Plot LSTM
if "LSTM" in avg_histories and avg_histories["LSTM"]["accuracy"].size > 0:
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(avg_histories["LSTM"]['accuracy'], label='LSTM Train Acc')
    plt.plot(avg_histories["LSTM"]['val_accuracy'], label='LSTM Val Acc')
    plt.title('LSTM Average Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(avg_histories["LSTM"]['loss'], label='LSTM Train Loss')
    plt.plot(avg_histories["LSTM"]['val_loss'], label='LSTM Val Loss')
    plt.title('LSTM Average Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.suptitle("LSTM Averaged Learning Curves Across Folds", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# Plot BERT
# BERT typically runs for fewer epochs. Check for valid data before plotting.
if "BERT" in avg_histories and avg_histories["BERT"]["accuracy"].size > 0:
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(avg_histories["BERT"]['accuracy'], label='BERT Train Acc')
    plt.plot(avg_histories["BERT"]['val_accuracy'], label='BERT Val Acc')
    plt.title('BERT Average Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(avg_histories["BERT"]['loss'], label='BERT Train Loss')
    plt.plot(avg_histories["BERT"]['val_loss'], label='BERT Val Loss')
    plt.title('BERT Average Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.suptitle("BERT Averaged Learning Curves Across Folds", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()